In [10]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
import os

In [11]:
import pandas as pd
import numpy as np
import geopandas as gpd

GEO_DATA_PATH = "../../../raw/geo_data/shapecountries.shp"
INPUT_FOLDER = "../../../clean"
OUTPUT_FOLDER = "../../../processed"


co2 = pd.read_csv(os.path.join(INPUT_FOLDER, "co2.csv")).rename(columns={"CO2": "co2"})
n2o = pd.read_csv(os.path.join(INPUT_FOLDER, "n2o.csv")).rename(columns={"N2O": "n2o"})
ch4 = pd.read_csv(os.path.join(INPUT_FOLDER, "ch4.csv")).rename(columns={"CH4": "ch4"})
ddcg = pd.read_csv(os.path.join(INPUT_FOLDER, "DDCGdata.csv")).query("year >= 1970")

geo_data = gpd.read_file(GEO_DATA_PATH)
geo_data["_ID"] = (geo_data.reset_index()["index"] + 1).astype(int)


print(ddcg["wbcode"].unique().shape)
merged = ddcg.merge(co2, "left")
print(merged["wbcode"].unique().shape)
print(merged.shape)
merged = merged.merge(ch4)
print(merged.shape)
merged = merged.merge(n2o)
print(merged.shape)

gdp_1970 = merged.query("year==1970")[["_ID", "gdp_pc"]]
gdp_1980 = merged.query("year==1980")[["_ID", "gdp_pc"]]
merged_geodata = geo_data.merge(gdp_1970, on="_ID").rename(columns={"gdp_pc": "gdp1970"})
merged_geodata = merged_geodata.merge(gdp_1980, on="_ID").rename(columns={"gdp_pc": "gdp1980"})

def impute_1970(row):
	if ~np.isnan(row["gdp1970"]):
		return row
	
	neighbors = merged_geodata[~merged_geodata.geometry.disjoint(row.geometry)]
	
	if np.all(neighbors["gdp1970"].isna()):
		return row
	
	row["gdp1970"] = np.nanmedian(neighbors["gdp1970"])
	return row

def impute_1980(row):
	if ~np.isnan(row["gdp1980"]):
		return row
	
	neighbors = merged_geodata[~merged_geodata.geometry.disjoint(row.geometry)]
	
	if np.all(neighbors["gdp1980"].isna()):
		return row
	
	row["gdp1980"] = np.nanmedian(neighbors["gdp1980"])
	return row

def imputation(row):
	row = impute_1970(row)
	row = impute_1980(row)
	return row

merged_geodata = merged_geodata.apply(imputation, axis=1)
merged = merged.merge(merged_geodata[["_ID", "gdp1970", "gdp1980"]], on="_ID")
print(merged.shape)

regions = pd.read_csv(os.path.join(INPUT_FOLDER, "wb-regions.csv"))
merged = merged.merge(regions).drop_duplicates()
print(merged.shape)

merged.to_stata(os.path.join(OUTPUT_FOLDER, "pollution.dta"))

(184,)
(184,)
(7544, 9)
(7544, 10)
(7544, 11)
(7544, 13)
(7544, 14)
